In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import UEG_response as ur
import json

# import matplot2tikz 

In [ ]:
# Units
hbar = 1.0
aB = 1.0
m = 1.0
e = 1.0

# Tolerances
reltol = 1e-16
abstol = 1e-8
eta_log  = 1e-6
eta_sqrt = 1e-6
eta_pol = 1e-4
points_n = 5
tol_upper = 1e-8
dx = 1e-4
lower = 1e-6
limit = 50


In [ ]:
# Comparison to DFT for different directions

# Setup of some parameters
rs_str = "3.23"
theta_str = "1.0"
N = 38
base_folder = 'data_external'

# Conditions
rs = float(rs_str)
theta = float(theta_str)

# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
beta = 1/(theta*EF)
n = 3/(4*np.pi*rs**3)
L = (N/n)**(1/3)
beta_eff = beta / np.sqrt(1 + (1/theta)**2 )

# sets
n1s = [ np.array([0, 0, 1]) for _ in range(5) ]
n2s = [ np.array([0, 0, 2]), np.array([0, 1, 1]), np.array([0, 2, 0]), np.array([0, 1, -1]), np.array([0, 0, -2]) ]
n3s = [n1 + n2 for n1, n2 in zip(n1s, n2s)]

n_vec = np.linspace(0.0, 6, 500)

# Load DFT data comparison
data_file = f"%s/extracted_responses.json"%(base_folder)
with open(data_file, "r") as f:
    serilized_dict = f.read()
response_dict = json.loads(serilized_dict)

# Plot settings
colors  = ['g', 'r', 'm', 'b', 'k']
markers = ['s', 'o', 'd', '^', 'v']

for i in range(len(n1s)):
    n1 = n1s[i]
    n2 = n2s[i]
    n3 = n3s[i]

    L = ( N / n )**(1/3) 
    k1 = 2*np.pi * n1 / np.array([L, L, L])
    k2 = 2*np.pi * n2 / np.array([L, L, L])
    k3 = 2*np.pi * n3 / np.array([L, L, L])

    costheta = np.dot(k1,k2)/(np.linalg.norm(k1)*np.linalg.norm(k2))

    chi2_0_12 = ur.ideal_quadratic_response(0.0, n_vec*np.linalg.norm(k1), 0.0, n_vec*np.linalg.norm(k2), costheta,
                                        m, hbar, n, beta, ms=2,
                                        reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                        dx=dx, points_n=points_n)

    n_DFT      = []
    chi2_0_DFT = []
    for nn in range(1,10):
        key = f"set_%d-%d"%(i+1,nn)
        if (key in response_dict):
            n_DFT.append(nn)
            chi2_0_DFT.append(response_dict[key][-1])

    if (len(n_DFT) >= 1):
        plt.plot(np.array(n_DFT)*2*np.pi/(L*qF), np.array(chi2_0_DFT)/(n*beta_eff**2), linestyle='None', marker=markers[i], color=colors[i], label=r"Set $%d$ (DFT)"%(i+1))

    if (colors[i] == 'k'):
        label = r"This work"
    else:
        label = None
    plt.plot(n_vec*np.linalg.norm(k1)/qF, np.real(chi2_0_12)/(n*beta_eff**2), color=colors[i], label=label)


plt.xlabel(r'$|\vec{k}_1|/q_F$')
plt.ylabel(r'$\chi^{(2)}_0(\vec{k}_1, \vec{k}_2)$ $[n\beta_{eff}^2]$')

plt.xlim(left=0.0)
plt.xlim(right=3.5)
plt.ylim(bottom=0.0)

plt.legend()

# plt.savefig(f"figures/DFT_test_directions.jpg", dpi=400, bbox_inches='tight')
# matplot2tikz.save("figures/DFT_test_directions.tex")


In [ ]:
# Setup of some parameters
rs_str = "3.23"
theta_strs = ["0.01", "0.5", "1.0", "2.0", "10.0"]
N = 38
additional_N_DFT = [64, 76]

# PIMC data
N_PIMC = 38
PIMC_folder = "./data_external"
PIMC_response_file = "QuadraticResponse_fermion_full_quadratic_density_response_fifth_%d.res"%(N_PIMC)


# Conditions
rs = float(rs_str)

# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
n = 3/(4*np.pi*rs**3)

# sets
n1s = [ np.array([0, 0, 1]) for _ in range(5) ]
n2s = [ np.array([0, 0, 2]), np.array([0, 1, 1]), np.array([0, 2, 0]), np.array([0, 1, -1]), np.array([0, 0, -2]) ]
n3s = [n1 + n2 for n1, n2 in zip(n1s, n2s)]

n_vec = np.linspace(0.0, 6.8, 500)

# Load DFT data comparison
data_file = f"%s/extracted_responses_temp.json"%(base_folder)
with open(data_file, "r") as f:
    serilized_dict = f.read()
response_dict = json.loads(serilized_dict)

# Plot settings
colors  = ['g', 'r', 'm', 'b', 'k']
markers = ['s', 'o', 'd', 'x', '.']

# Set index
set_idx = 3
set_str = ("_".join([ str(nn) for nn in n2s[set_idx]])).replace('-', 'n')

for i, theta_str in enumerate(theta_strs):
    # Conditions
    theta = float(theta_str)
    beta = 1/(theta*EF)

    n1 = n1s[set_idx]
    n2 = n2s[set_idx]
    n3 = n3s[set_idx]

    L = ( N / n )**(1/3) 
    k1 = 2*np.pi * n1 / np.array([L, L, L])
    k2 = 2*np.pi * n2 / np.array([L, L, L])
    k3 = 2*np.pi * n3 / np.array([L, L, L])

    costheta = np.dot(k1,k2)/(np.linalg.norm(k1)*np.linalg.norm(k2))

    chi2_0_12  = ur.ideal_quadratic_response(0.0, n_vec*np.linalg.norm(k1), 0.0, n_vec*np.linalg.norm(k2), costheta,
                                            m, hbar, n, beta, ms=2,
                                            reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                            dx=dx, points_n=points_n, force_output=True)
    
    beta_eff = beta / np.sqrt(1 + (2/(3*theta))**2 )
    norm = (n*beta_eff**2)

    k_DFT      = []
    chi2_0_DFT = []

    for nn in range(1,10):
        key = f"t_%s_set_%d-%d"%(theta_str.replace('.', '_'),set_idx+1,nn)
        if (key in response_dict):
            k_DFT.append(nn*2*np.pi/(L))
            chi2_0_DFT.append(response_dict[key][-1])

    for N_DFT in additional_N_DFT:
        for nn in range(1,10):
            key = f"%s_t_%s_set_%d-%d"%(f"N%d"%(N_DFT),theta_str.replace('.', '_'),set_idx+1,nn)
            if (key in response_dict):
                L_DFT = ( N_DFT / n )**(1/3)
                k_DFT.append(nn*2*np.pi/(L_DFT))
                chi2_0_DFT.append(response_dict[key][-1])    

    if (len(k_DFT) >= 1):
        plt.plot(np.array(k_DFT)/qF, np.array(chi2_0_DFT)/norm, linestyle='None', marker=markers[i], color=colors[i], markerfacecolor='None', label=r"$\Theta = %s$ (DFT)"%(theta_str))

    # Plot PIMC data
    PIMC_file = f"%s/ideal_rs_%s_theta_%s_N_%d_xi_1.0_P_200_fixed_config_%s/%s"%(PIMC_folder,rs_str,theta_str,N_PIMC,set_str,PIMC_response_file)
    if (os.path.isfile(PIMC_file)):
        PIMC_data = np.loadtxt(PIMC_file)
        k1_mag       = np.sqrt( np.sum(PIMC_data[:, :3]**2, axis=1) )
        k2_mag       = np.sqrt( np.sum(PIMC_data[:,3:6]**2, axis=1) )
        L_PIMC       = (N_PIMC/n)**(1/3)
        chi0_2       = PIMC_data[:, 6]/(2*L_PIMC**3)
        sigma_chi0_2 = PIMC_data[:, 7]/(2*L_PIMC**3)

        plt.errorbar(k1_mag/qF, chi0_2/norm, yerr=1.96*sigma_chi0_2/norm, linestyle='None', marker=markers[i], capsize=2, color=colors[i], label=r"$\Theta = %s$ (PIMC)"%(theta_str))


    if (colors[i] == 'k'):
        label = r"This work"
    else:
        label = None
    plt.plot(n_vec*np.linalg.norm(k1)/qF, np.real(chi2_0_12)/norm, color=colors[i], label=label)

# T = 0
n_vec = n_vec[1:]
ground_state_chi2_0 = ur.ground_state_ideal_quadratic_response(0.0, n_vec*np.linalg.norm(k1), 0.0, n_vec*np.linalg.norm(k2), costheta, m, hbar, n, ms=2)

norm = n*(3/(2*EF))**2
plt.plot(n_vec*np.linalg.norm(k1)/qF, np.real(ground_state_chi2_0)/norm, ':', color='k', label=r"$T = 0$")


plt.xlabel(r'$|\vec{k}_1|/q_F$')
plt.ylabel(r'$\chi^{(2)}_0(\vec{k}_1, \vec{k}_2)$ $[n\bar{\beta}_{eff}^2]$')

plt.xlim(left=0.0)
plt.xlim(right=3.5)
plt.ylim(bottom=0.0)
# plt.ylim(top=0.70)

plt.legend(ncol=2, loc='lower left')

# plt.savefig(f"figures/DFT_test_temp.jpg", dpi=400, bbox_inches='tight')
# matplot2tikz.save("figures/DFT_test_temp.tex")


In [ ]:
# Conditions
rs_str    = "3.23"
theta_str = "4.0"
rs = float(rs_str)
theta = float(theta_str)
inv_theta = 1/theta
N = 14
P = 200

log_plot = False


# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
beta = 1/(theta*EF)
n = 3/(4*np.pi*rs**3)
L = (N/n)**(1/3)

# Tolerances
reltol = 1e-12
abstol = 1e-8
eta_log  = 1e-7
eta_sqrt = 1e-7
eta_pol = 1e-5
points_n = 5
tol_upper = 1e-8
s = 1/2
dx = 1e-4
lower = 1e-6
force_output = True



# Settings
processed_folder = "./data_external"
base_name = "grand_canonical_ideal_rs_%s_theta_%s_N_%d_xi_1.0_P_%d_"%(rs_str,theta_str,N,P)
file_name = "fermion_N_88888888.res"

k = 1
# Average N and the statistical uncertanty.
name = f"%s/%s/%s"%(processed_folder,base_name,file_name)
data = np.loadtxt(name)
As = [0.0]
N_ave = [np.sum(data[:, 1])]
sigma_N_ave = [np.sqrt(np.sum(data[:, 2]**2))]

k_index = 1
k1 = np.array([k_index*2*np.pi/L])
A_strs = ["0.02", "0.04", "0.07", "0.14", "0.25", "0.35", "0.50", "0.70"]

for A_str in A_strs:
    add_name = "k_%d_A_%s"%(k_index,A_str.replace(".", "_"))
    name = f"%s/%s%s/%s"%(processed_folder,base_name,add_name,file_name)
    data = np.loadtxt(name)

    As.append(float(A_str))
    N_ave.append(np.sum(data[:, 1]))
    sigma_N_ave.append(np.sqrt(np.sum(data[:, 2]**2)))

As = np.array(As)
N_ave = np.array(N_ave)
sigma_N_ave = np.array(sigma_N_ave)

if (log_plot):
    As = As[1:]
    N_ave = N_ave[1:] - N_ave[0]
    sigma_N_ave = np.sqrt(sigma_N_ave[1:]**2 + sigma_N_ave[0]**2)


plt.errorbar(beta*As, N_ave, yerr=1.96*sigma_N_ave, marker="s", color='r', linestyle='None', capsize=3, label=r"Grand canonical PIMC: $|\vec{q}| = %.3g\, q_F$"%(k1[0]/qF))

# Plot model for zeroth
# chi2_zeroth = _chi0_k2_0_Maldague(k1/qF, qF, eta, beta, hbar, m, lower, reltol, abstol, tol_upper, points_n, s, force_output=True)

chi2_zeroth = ur.ideal_quadratic_response(0.0, k1, 0.0, 0.0, 1.0,
                                            m, hbar, n, beta, ms=2,
                                            reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                            dx=dx, points_n=points_n, force_output=True)

if not (log_plot):
    A_fine = np.linspace(0.0, 1.05*np.max(As))
    N_model = N + 2 * L**3 * np.real(chi2_zeroth) * A_fine**2
else:
    A_fine = np.logspace(-2, np.log10(1.05*np.max(As)), 100)
    N_model = 2 * L**3 * np.real(chi2_zeroth) * A_fine**2

plt.plot(beta*A_fine, N_model, '--r', label=r"Model: $|\vec{q}| = %.3g\, q_F$"%(k1[0]/qF))


# k = 5
# Average N and the statistical uncertanty.
name = f"%s/%s/%s"%(processed_folder,base_name,file_name)
data = np.loadtxt(name)
As = [0.0]
N_ave = [np.sum(data[:, 1])]
sigma_N_ave = [np.sqrt(np.sum(data[:, 2]**2))]

k_index = 5
k1 = np.array([k_index*2*np.pi/L])
A_strs = ["0.02", "0.04", "0.07", "0.14", "0.25", "0.35", "0.50", "0.70"]

for A_str in A_strs:
    add_name = "k_%d_A_%s"%(k_index,A_str.replace(".", "_"))
    name = f"%s/%s%s/%s"%(processed_folder,base_name,add_name,file_name)
    data = np.loadtxt(name)

    As.append(float(A_str))
    N_ave.append(np.sum(data[:, 1]))
    sigma_N_ave.append(np.sqrt(np.sum(data[:, 2]**2)))

As = np.array(As)
N_ave = np.array(N_ave)
sigma_N_ave = np.array(sigma_N_ave)

if (log_plot):
    As = As[1:]
    N_ave = N_ave[1:] - N_ave[0]
    sigma_N_ave = np.sqrt(sigma_N_ave[1:]**2 + sigma_N_ave[0]**2)

plt.errorbar(beta*As, N_ave, yerr=1.96*sigma_N_ave, marker="d", color='k', linestyle='None', capsize=3, label=r"Grand canonical PIMC: $|\vec{q}| = %.3g\, q_F$"%(k1[0]/qF))

# Plot model for zeroth
chi2_zeroth = ur.ideal_quadratic_response(0.0, k1, 0.0, 0.0, 1.0,
                                            m, hbar, n, beta, ms=2,
                                            reltol=reltol, abstol=abstol, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                            dx=dx, points_n=points_n, force_output=True)

if not (log_plot):
    A_fine = np.linspace(0.0, 1.05*np.max(As))
    N_model = N + 2 * L**3 * np.real(chi2_zeroth) * A_fine**2
else:
    A_fine = np.logspace(-2, np.log10(1.05*np.max(As)), 100)
    N_model = 2 * L**3 * np.real(chi2_zeroth) * A_fine**2

plt.plot(beta*A_fine, N_model, '-.k', label=r"Model: $|\vec{q}| = %.3g\, q_F$"%(k1[0]/qF))

if (log_plot):
    plt.yscale('log')
    plt.xscale('log')
    plt.ylabel(r"$\langle N \rangle_A - \langle N \rangle_0$")
    plt.xlim(left=1e-2)
else:
    plt.ylabel(r"$\langle N \rangle_A $")
    plt.xlim(left=-0.0005)

plt.xlabel(r"$\beta A$")
plt.legend()


# plt.savefig(f"figures/zeroth_harmonic_denisty_shift.jpg", dpi=400, bbox_inches='tight')
# matplot2tikz.save("figures/zeroth_harmonic_denisty_shift.tex")
